In [39]:
from google.colab import auth
auth.authenticate_user()
import polars as pl
from datasets import load_dataset, Features, Value
import os
import json

In [40]:
# --- 1. Cấu hình đường dẫn GCS ---
PROJECT_ID = "project-7f16ff1d-9026-4799-bfa"
BUCKET_NAME = "amazon-reviews-lakehouse-data"
listCategory = [
    'Automotive', 'Baby_Products', 'Beauty_and_Personal_Care', 'Books',
    'CDs_and_Vinyl', 'Cell_Phones_and_Accessories', 'Clothing_Shoes_and_Jewelry',
    'Digital_Music', 'Electronics', 'Gift_Cards', 'Grocery_and_Gourmet_Food',
    'Health_and_Household', 'Home_and_Kitchen', 'Industrial_and_Scientific',
    'Kindle_Store', 'Luxury_Beauty', 'Magazine_Subscriptions', 'Movies_and_TV',
    'Musical_Instruments', 'Office_Products', 'Patio_Lawn_and_Garden',
    'Pet_Supplies', 'Prime_Pantry', 'Software', 'Sports_and_Outdoors',
    'Subscription_Boxes', 'Tools_and_Home_Improvement', 'Toys_and_Games',
    'Video_Games'
]

In [41]:
def load_meta_data(CATEGORY, batch=100000):
    GCS_SAVE_PATH = f"gs://{BUCKET_NAME}/bronze-zone/meta_data/{CATEGORY}"

    # CHIẾN THUẬT: Đọc dưới dạng "text" để bỏ qua hoàn toàn việc kiểm tra Schema của Arrow
    # Cách này sẽ không bao giờ bị lỗi "Couldn't cast" hay "column names don't match"
    dataset = load_dataset(
        "text",
        data_files=f"https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/meta_categories/meta_{CATEGORY}.jsonl",
        split="train",
        streaming=True
    )

    BATCH_SIZE = batch
    buffer = []
    # Danh sách thuộc tính cần giữ lại
    META_KEYS = ['parent_asin', 'title', 'main_category', 'price', 'store', 'average_rating', 'rating_number']

    print(f"🚀 [Text-Parsing Mode] Đang xử lý Metadata {CATEGORY}...")

    try:
        for i, row in enumerate(dataset):
            # 1. Giải mã chuỗi JSON từ dòng văn bản
            try:
                raw_record = json.loads(row['text'])
            except json.JSONDecodeError:
                continue # Bỏ qua nếu dòng đó bị lỗi định dạng JSON

            # 2. Chỉ trích xuất các trường mục tiêu và ép về String để Bronze bền bỉ
            record = {}
            for k in META_KEYS:
                val = raw_record.get(k)
                # Ép mọi thứ về string để tránh lỗi kiểu dữ liệu hỗn hợp ở tầng Bronze
                record[k] = str(val) if val is not None else None

            buffer.append(record)

            # 3. Ghi dữ liệu theo Batch
            if len(buffer) == BATCH_SIZE:
                df_batch = pl.DataFrame(buffer)
                batch_num = (i + 1) // BATCH_SIZE
                file_name = f"{GCS_SAVE_PATH}/batch_{batch_num}.parquet"

                df_batch.write_parquet(file_name, use_pyarrow=True)
                print(f"✅ Đã lưu Batch {batch_num} ({i+1} dòng)")
                buffer = []

        # Xử lý Batch cuối cùng
        if buffer:
            final_batch_num = (i // BATCH_SIZE) + 1
            final_file = f"{GCS_SAVE_PATH}/batch_{final_batch_num}_final.parquet"
            pl.DataFrame(buffer).write_parquet(final_file, use_pyarrow=True)
            print(f"✅ Đã lưu Batch cuối cùng cho {CATEGORY}.")

    except Exception as e:
        print(f"❌ Lỗi nghiêm trọng tại danh mục {CATEGORY}: {e}")

    print(f"🏁 Hoàn thành: {GCS_SAVE_PATH}")

In [42]:
def read_meta_data(CATEGORY,batch=1000):
  print(f'Du lieu {CATEGORY}')
  path = f"gs://amazon-reviews-lakehouse-data/bronze-zone/meta_data/{CATEGORY}/*.parquet"
  lazy_df = pl.scan_parquet(path)
  query = lazy_df.head(batch)
  df = query.collect()
  print(df)

In [ ]:
for category in listCategory:
  load_meta_data(category,200000)

🚀 [Text-Parsing Mode] Đang xử lý Metadata Automotive...
✅ Đã lưu Batch 1 (200000 dòng)
✅ Đã lưu Batch 2 (400000 dòng)
✅ Đã lưu Batch 3 (600000 dòng)
✅ Đã lưu Batch 4 (800000 dòng)
✅ Đã lưu Batch 5 (1000000 dòng)
✅ Đã lưu Batch 6 (1200000 dòng)
✅ Đã lưu Batch 7 (1400000 dòng)
✅ Đã lưu Batch 8 (1600000 dòng)
✅ Đã lưu Batch 9 (1800000 dòng)
✅ Đã lưu Batch 10 (2000000 dòng)
✅ Đã lưu Batch cuối cùng cho Automotive.
🏁 Hoàn thành: gs://amazon-reviews-lakehouse-data/bronze-zone/meta_data/Automotive
🚀 [Text-Parsing Mode] Đang xử lý Metadata Baby_Products...
✅ Đã lưu Batch 1 (200000 dòng)
✅ Đã lưu Batch cuối cùng cho Baby_Products.
🏁 Hoàn thành: gs://amazon-reviews-lakehouse-data/bronze-zone/meta_data/Baby_Products
🚀 [Text-Parsing Mode] Đang xử lý Metadata Beauty_and_Personal_Care...
✅ Đã lưu Batch 1 (200000 dòng)
✅ Đã lưu Batch 2 (400000 dòng)
✅ Đã lưu Batch 3 (600000 dòng)
✅ Đã lưu Batch 4 (800000 dòng)
✅ Đã lưu Batch 5 (1000000 dòng)
✅ Đã lưu Batch cuối cùng cho Beauty_and_Personal_Care.
🏁 Ho

In [43]:
read_meta_data('Arts_Crafts_and_Sewing')

Du lieu Arts_Crafts_and_Sewing
shape: (1_000, 7)
┌─────────────┬───────────────┬───────────────┬───────┬──────────────┬──────────────┬──────────────┐
│ parent_asin ┆ title         ┆ main_category ┆ price ┆ store        ┆ average_rati ┆ rating_numbe │
│ ---         ┆ ---           ┆ ---           ┆ ---   ┆ ---          ┆ ng           ┆ r            │
│ str         ┆ str           ┆ str           ┆ str   ┆ str          ┆ ---          ┆ ---          │
│             ┆               ┆               ┆       ┆              ┆ f64          ┆ i64          │
╞═════════════╪═══════════════╪═══════════════╪═══════╪══════════════╪══════════════╪══════════════╡
│ B013W3MPCW  ┆ Trapp Home    ┆ Amazon Home   ┆ null  ┆ Trapp        ┆ 4.4          ┆ 108          │
│             ┆ Fragrance Wax ┆               ┆       ┆              ┆              ┆              │
│             ┆ Melts…        ┆               ┆       ┆              ┆              ┆              │
│ B09B59XWTG  ┆ JESEP YONG    ┆ Arts, Craf